# 03 · Gold · `fato_atendimento`
### V-Energy Grid 360 — Arquitetura Medalhão

**Status: executada no Databricks, com todas as checagens `PASS` (36).** Fecha com a Silver (318.871 chamados; soma de `tempo_resolucao_horas` idêntica) e reproduz os números da exploração: 9.566 chamados sem nota, 17 pares assunto × subassunto, 18.061 chamados em 2025-07 e +16,0% entre os dois períodos. Conexão com a `gold.dim_uc` testada: 0 UCs fora da dimensão e 0 com município, bairro ou classe diferentes.

Reúne, por chamado, o que a visão 360 e o agente precisam saber sobre o atendimento: onde a UC fica, o assunto, o tempo até fechar e a nota. Grava `v_energy.gold.fato_atendimento`, seguindo o **contrato da Gold (tabela B1, versão v4)**, sem mudança de colunas.

**Grão:** 1 linha = 1 chamado (`protocolo`). Esperado: **318.871 linhas**, com `protocolo` único.

**Quem usa:** visão 360 ("os chamados que o cliente já abriu", da Camila), KPIs mensais (`kpis_mensal`: `qtd_atendimentos` e `nota_satisfacao_media`) e o agente de IA.

**Fontes (Silver):** `atendimentos`, `tratativas_atendimento` (só para `ultima_acao`) e `unidades_consumidoras` (município, bairro e classe da UC).

| # | Passo | Regra |
|---|---|---|
| 1 | Chamado | `silver.atendimentos`, já tratada: datas em `timestamp`, nota em `decimal(3,1)`, categorias padronizadas |
| 2 | Datas | `data_abertura` e `data_fechamento` viram `data_hora_abertura` e `data_hora_fechamento`, **com horário**: sem cast para `date` (pedido de Dev no contrato, e o `tempo_resolucao_horas` precisa da hora) |
| 3 | Localização da UC | `municipio`, `bairro` e `classe` de `silver.unidades_consumidoras` |
| 4 | `ultima_acao` | ação da tratativa de **maior `sequencia`** de cada chamado, em `silver.tratativas_atendimento` |
| 5 | Resto | `mes_referencia`, `aberto`, `tempo_resolucao_horas`, `qtd_tratativas` e `nota_satisfacao` já vêm da Silver |

**O que fica fora da Gold:** as tratativas (decisão da v4). A visão 360 só precisa de `qtd_tratativas` e `ultima_acao`. Se a tela de detalhe do chamado precisar do histórico completo, a `silver.tratativas_atendimento` pode virar uma tabela Gold depois, sem mudar esta.

**Por que a tabela passa por uma tabela temporária (`_stg`):** o Spark serverless não permite `.cache()`, e cada checagem recalcularia as junções (chamados, tratativas e UCs). O resultado é gravado primeiro em `gold.fato_atendimento_stg`; as checagens leem dela; **só se todas passarem** a tabela final é gravada e a temporária é apagada.

**O que entra no dicionário de dados desta tabela:**
- **`ultima_acao` é a última ação registrada, e não a situação do chamado** (log A5): as cinco ações se dividem em partes iguais, e a primeira tratativa nem sempre é `ABERTURA`. Para saber se o chamado está aberto, use `aberto`;
- **`aberto` é falso em todos os chamados desta base**: não há chamados em aberto. O **histórico de chamados do cliente** (todos os que ele já abriu) aparece normalmente na visão 360;
- **pico de julho de 2025:** 18.061 chamados, cerca de 2,8× o normal, no mesmo mês de um evento climático na rede. O tempo até fechar sobe de ~120 h para ~361 h e a nota cai de ~4,10 para ~3,11. Esse mês explica quase toda a alta de chamados entre os dois períodos do diagnóstico (+16%; sem o excesso de julho, ~0,8%) (log A6);
- **9.566 chamados sem nota (3,0%):** o cliente não avaliou; a média usa só as notas preenchidas, na escala de 1 a 5 (log A1);
- **acúmulo de notas em 5,0** (6.160, contra 4.363 em 4,9): hipótese de corte, sem correção (log A2);
- os **17 pares assunto × subassunto** saem desta tabela (`SELECT DISTINCT assunto, subassunto`) e vão para o dicionário, para o Dev montar filtros dependentes.

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from pyspark.sql import functions as F, Row, Window
from datetime import datetime

tabela_atend = f"{silver_schema}.atendimentos"
tabela_trat = f"{silver_schema}.tratativas_atendimento"
tabela_uc = f"{silver_schema}.unidades_consumidoras"
tabela_stg = f"{gold_schema}.fato_atendimento_stg"
tabela_gold = f"{gold_schema}.fato_atendimento"
tabela_dq_log = f"{silver_schema}.dq_log"      # mesmo log das Silvers e das outras Golds

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
print(f"origem:  {tabela_atend}, {tabela_trat}, {tabela_uc}")
print(f"destino: {tabela_gold} (via {tabela_stg})")

origem:  v_energy.silver.atendimentos, v_energy.silver.tratativas_atendimento, v_energy.silver.unidades_consumidoras
destino: v_energy.gold.fato_atendimento (via v_energy.gold.fato_atendimento_stg)


## 2. Funções de checagem de qualidade

Mesmas funções da aula e das Silvers (`dq_check`, `dq_check_unique` e `dq_check_contagem`), com as mesmas colunas no log (`table_name`, `check_name`, `total_rows`, `failed_rows`, `passed`, `checked_at`). Acrescento `dq_check_valor`, para comparar um número calculado com um esperado (somas que precisam bater com as Silvers e com o diagnóstico); ela grava no log nas mesmas colunas, com `failed_rows` igual a 0 ou 1.

As funções só acumulam os resultados em `dq_results`. Depois das checagens, o notebook **grava o resultado em `silver.dq_log`** (sempre) e **só então decide se grava a Gold**.

**Cuidado com nulos:** linhas em que a condição resulta em nulo não contam como falha. Quando o nulo também for um problema, há um `dq_check` separado com `isNotNull()`.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def n_br(n):
    return f"{n:,}".replace(",", ".")

def dq_check_contagem(table_name: str, check_name: str, df, esperado: int):
    """Checagem de contagem total: passa só se o número de linhas for exatamente o esperado."""
    total = df.count()
    diferenca = abs(total - esperado)
    passed = diferenca == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=diferenca, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {n_br(total)} linhas (esperado {n_br(esperado)})")

def dq_check_valor(table_name: str, check_name: str, total_rows: int, obtido, esperado):
    """Compara um valor calculado com o esperado (soma que precisa bater com a Silver ou com o diagnóstico)."""
    passed = obtido == esperado
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total_rows,
            failed_rows=0 if passed else 1, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | obtido {obtido} (esperado {esperado})")

## 3. Leitura das Silvers e conferência das colunas

Antes de construir, o notebook confere que as três Silvers existem **com as colunas de que precisa**; se faltar alguma, para aqui com uma mensagem clara.

Contagens esperadas:
- `silver.atendimentos`: **318.871** chamados (`protocolo` único);
- `silver.tratativas_atendimento`: **636.857** tratativas, de 1 a 3 por chamado;
- `silver.unidades_consumidoras`: **45.000** UCs.

In [0]:
atend = spark.table(tabela_atend)
trat = spark.table(tabela_trat)
uc = spark.table(tabela_uc)

exigidas = {
    tabela_atend: ["protocolo", "id_uc", "canal", "assunto", "subassunto", "data_abertura", "data_fechamento",
                   "mes_referencia", "aberto", "tempo_resolucao_horas", "qtd_tratativas", "nota_satisfacao"],
    tabela_trat: ["protocolo", "sequencia", "acao"],
    tabela_uc: ["id_uc", "classe", "bairro", "municipio"],
}
for tabela, colunas in exigidas.items():
    faltando = [c for c in colunas if c not in spark.table(tabela).columns]
    if faltando:
        raise Exception(f"{tabela} não tem as colunas {faltando}. Ajuste os nomes antes de seguir (DESCRIBE {tabela}).")

n_atend, n_trat, n_uc = atend.count(), trat.count(), uc.count()
print(f"chamados: {n_br(n_atend)} | tratativas: {n_br(n_trat)} | UCs: {n_br(n_uc)}")   # esperado: 318.871 | 636.857 | 45.000
atend.printSchema()

chamados: 318.871 | tratativas: 636.857 | UCs: 45.000
root
 |-- protocolo: string (nullable = true)
 |-- id_uc: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- assunto: string (nullable = true)
 |-- subassunto: string (nullable = true)
 |-- data_abertura: timestamp (nullable = true)
 |-- data_fechamento: timestamp (nullable = true)
 |-- mes_referencia: string (nullable = true)
 |-- aberto: boolean (nullable = true)
 |-- tempo_resolucao_horas: decimal(10,2) (nullable = true)
 |-- qtd_tratativas: integer (nullable = true)
 |-- nota_satisfacao: decimal(3,1) (nullable = true)
 |-- source_file: string (nullable = true)



## 4. Montagem da `fato_atendimento`

**Decisão:** partir de `silver.atendimentos` e acrescentar, com `left join` (nenhum chamado se perde), a localização da UC e a `ultima_acao`.

**`ultima_acao`:** para cada `protocolo`, a `acao` da tratativa de **maior `sequencia`**, com `max_by(acao, sequencia)`. Na Silver, `sequencia` é a posição na lista mais 1 (verificado em 100%), então a de maior valor é a última registrada.

**Datas com horário:** as colunas só mudam de nome (`data_abertura` → `data_hora_abertura` e `data_fechamento` → `data_hora_fechamento`). O log A4 dizia que a Gold faria o cast para `date`, mas o contrato pede **data e hora** desde a v3. **Não há cast.**

**Atributos da UC:** `municipio`, `bairro` e `classe` são copiados para facilitar filtros e a visão 360. A fonte de verdade continua sendo a `dim_uc`, e o bairro só vale junto com o município.

**Ordem das colunas:** a do contrato (16 colunas).

**Conferência:** 318.871 linhas depois das junções, 0 sem UC e 0 sem `ultima_acao`.

In [0]:
# Montagem (esperado: 318.871 linhas; 0 sem localização; 0 sem ultima_acao)
ultima = trat.groupBy("protocolo").agg(F.expr("max_by(acao, sequencia)").alias("ultima_acao"))
uc_sel = uc.select("id_uc", "municipio", "bairro", "classe")

df_fato = (
    atend
    .join(F.broadcast(uc_sel), "id_uc", "left")
    .join(ultima, "protocolo", "left")
    .select(
        "protocolo", "id_uc", "municipio", "bairro", "classe",
        "canal", "assunto", "subassunto",
        F.col("data_abertura").alias("data_hora_abertura"),
        F.col("data_fechamento").alias("data_hora_fechamento"),
        "mes_referencia", "aberto", "tempo_resolucao_horas", "qtd_tratativas",
        "ultima_acao", "nota_satisfacao",
    )
)
print("colunas:", len(df_fato.columns))     # esperado: 16
df_fato.printSchema()

# Grava na tabela temporária; as checagens leem dela
(
    df_fato.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_stg)
)
fato = spark.table(tabela_stg)
print(f"{tabela_stg}: {n_br(fato.count())} linhas")                                         # esperado: 318.871
print("sem localização da UC:", n_br(fato.filter("municipio IS NULL OR bairro IS NULL OR classe IS NULL").count()))   # esperado: 0
print("sem ultima_acao:", n_br(fato.filter("ultima_acao IS NULL").count()))                  # esperado: 0
display(fato.limit(5))

colunas: 16
root
 |-- protocolo: string (nullable = true)
 |-- id_uc: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- bairro: string (nullable = true)
 |-- classe: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- assunto: string (nullable = true)
 |-- subassunto: string (nullable = true)
 |-- data_hora_abertura: timestamp (nullable = true)
 |-- data_hora_fechamento: timestamp (nullable = true)
 |-- mes_referencia: string (nullable = true)
 |-- aberto: boolean (nullable = true)
 |-- tempo_resolucao_horas: decimal(10,2) (nullable = true)
 |-- qtd_tratativas: integer (nullable = true)
 |-- ultima_acao: string (nullable = true)
 |-- nota_satisfacao: decimal(3,1) (nullable = true)

v_energy.gold.fato_atendimento_stg: 318.871 linhas
sem localização da UC: 0
sem ultima_acao: 0


protocolo,id_uc,municipio,bairro,classe,canal,assunto,subassunto,data_hora_abertura,data_hora_fechamento,mes_referencia,aberto,tempo_resolucao_horas,qtd_tratativas,ultima_acao,nota_satisfacao
AT000000000,UC142511,Recife,Monteiro,COMERCIAL,SITE,RELIGACAO,URGENCIA,2022-07-02T15:51:00.000Z,2022-07-10T14:51:00.000Z,2022-07,false,191.00,1,RETORNO AO CLIENTE,3.9
AT000000001,UC100661,Recife,Paissandu,RESIDENCIAL,SITE,FALTA DE ENERGIA,INTERRUPCAO TOTAL,2022-07-07T23:25:00.000Z,2022-07-17T10:25:00.000Z,2022-07,false,227.00,2,DESPACHO,4.1
AT000000002,UC108189,Olinda,Tabajara,RESIDENCIAL,WHATSAPP,LIGACAO NOVA,SOLICITACAO,2022-07-11T10:48:00.000Z,2022-07-13T21:48:00.000Z,2022-07,false,59.00,3,ENCERRAMENTO,4.4
AT000000003,UC104975,Recife,Arruda,RESIDENCIAL,TELEFONE,DANO ELETRICO,VISTORIA,2022-07-09T17:11:00.000Z,2022-07-16T05:11:00.000Z,2022-07,false,156.00,3,RETORNO AO CLIENTE,4.0
AT000000004,UC108340,Paulista,Jardim Paulista,RESIDENCIAL,WHATSAPP,FATURA,SEGUNDA VIA,2022-07-23T15:44:00.000Z,2022-07-27T19:44:00.000Z,2022-07,false,100.00,2,ENCERRAMENTO,4.0


## 5. Checagens de qualidade

Rodam sobre a tabela temporária, **antes** de gravar a final:

| Checagem | Protege |
|---|---|
| Total de linhas = 318.871; `protocolo` único; igual à Silver | junção que perde ou multiplica chamados |
| Colunas-chave não nulas (inclui localização e `ultima_acao`) | junções sem correspondência |
| `protocolo` = `AT` + 9 dígitos; `id_uc` = `UC` + 6 dígitos | formato das chaves |
| `canal`, `assunto`, `subassunto`, `classe` e `ultima_acao` nos valores do contrato | domínio das categorias |
| `aberto` = fechamento nulo; abertura <= fechamento; `tempo_resolucao_horas` >= 0 | datas |
| `mes_referencia` = mês da abertura | derivado |
| Nota entre 1 e 5 (quando preenchida); **9.566 notas nulas** | a nota |
| `qtd_tratativas` entre 1 e 3 e igual ao número de tratativas da Silver | integridade com o histórico |
| **17 pares assunto × subassunto** | os pares do contrato |
| **Pico de julho de 2025 = 18.061 chamados** | o evento que o diagnóstico usa |
| Soma de `tempo_resolucao_horas` igual à da Silver | a Gold não alterou nada |

Os valores esperados valem para esta extração; numa reexecução com dados novos, precisam ser atualizados.

In [0]:
t = "gold.fato_atendimento"
dq_results.clear()   # se esta célula for reexecutada, não acumula resultados antigos

CANAIS = ["TELEFONE", "APP", "WHATSAPP", "SITE", "PRESENCIAL"]
ASSUNTOS = ["FALTA DE ENERGIA", "FATURA", "DANO ELETRICO", "LIGACAO NOVA", "GERACAO DISTRIBUIDA", "RELIGACAO"]
SUBASSUNTOS = ["VISTORIA", "INTERRUPCAO PARCIAL", "INTERRUPCAO TOTAL", "OSCILACAO", "URGENCIA", "RESSARCIMENTO",
               "APOS PAGAMENTO", "SOLICITACAO", "HOMOLOGACAO", "PRAZO", "CREDITOS", "CONTESTACAO",
               "SEGUNDA VIA", "VALOR ALTO", "PARCELAMENTO"]
ACOES = ["ABERTURA", "RETORNO AO CLIENTE", "DESPACHO", "ENCERRAMENTO", "ANALISE"]
CLASSES = ["RESIDENCIAL", "RURAL", "COMERCIAL", "PODER PUBLICO", "INDUSTRIAL"]

def no_dominio(coluna, valores):
    return F.coalesce(F.col(coluna).isin(*valores), F.lit(False))   # nulo também falha

# Volume e chave
dq_check_contagem(t, "total de chamados = 318.871", fato, 318_871)
dq_check_unique(t, "protocolo único", fato, ["protocolo"])
n_fato = fato.count()

# Nulos (data_hora_fechamento, tempo_resolucao_horas e nota_satisfacao podem ser nulos)
for coluna in ["protocolo", "id_uc", "municipio", "bairro", "classe", "canal", "assunto", "subassunto",
               "data_hora_abertura", "mes_referencia", "aberto", "qtd_tratativas", "ultima_acao"]:
    dq_check(t, f"{coluna} não nulo", fato, F.col(coluna).isNotNull())

# Formato das chaves e domínios
dq_check(t, "protocolo no padrão AT + 9 dígitos", fato, F.col("protocolo").rlike("^AT[0-9]{9}$"))
dq_check(t, "id_uc no padrão UC + 6 dígitos", fato, F.col("id_uc").rlike("^UC[0-9]{6}$"))
dq_check(t, "mes_referencia no formato yyyy-MM", fato, F.col("mes_referencia").rlike("^[0-9]{4}-[0-9]{2}$"))
dq_check(t, "canal só nos 5 valores", fato, no_dominio("canal", CANAIS))
dq_check(t, "assunto só nos 6 valores", fato, no_dominio("assunto", ASSUNTOS))
dq_check(t, "subassunto só nos 15 valores", fato, no_dominio("subassunto", SUBASSUNTOS))
dq_check(t, "ultima_acao só nas 5 ações", fato, no_dominio("ultima_acao", ACOES))
dq_check(t, "classe só nas 5 classes", fato, no_dominio("classe", CLASSES))

# Datas
dq_check(t, "aberto = data_hora_fechamento nula", fato, F.col("aberto") == F.col("data_hora_fechamento").isNull())
dq_check(t, "abertura <= fechamento (quando há fechamento)", fato,
         F.col("data_hora_fechamento").isNull() | (F.col("data_hora_abertura") <= F.col("data_hora_fechamento")))
dq_check(t, "mes_referencia = mês da data_hora_abertura", fato,
         F.col("mes_referencia") == F.date_format("data_hora_abertura", "yyyy-MM"))
dq_check(t, "tempo_resolucao_horas >= 0 e preenchido nos chamados fechados", fato,
         F.col("aberto") | F.coalesce(F.col("tempo_resolucao_horas") >= 0, F.lit(False)))
# as datas guardam o horário (contrato: sem cast para data): tem que haver aberturas fora da meia-noite
dq_check_valor(t, "datas com horário (há aberturas com hora e minuto)", n_fato,
               fato.filter("hour(data_hora_abertura) <> 0 OR minute(data_hora_abertura) <> 0").count() > 0, True)

# Nota
dq_check(t, "nota entre 1 e 5 (quando preenchida)", fato,
         F.col("nota_satisfacao").isNull() | F.col("nota_satisfacao").between(1, 5))
dq_check_valor(t, "chamados sem nota = 9.566", n_fato, fato.filter("nota_satisfacao IS NULL").count(), 9_566)

# Integridade com o histórico (silver.tratativas_atendimento)
trat_n = trat.groupBy("protocolo").agg(F.count("*").alias("n_trat"))
fato_trat = fato.join(trat_n, "protocolo", "left")
dq_check(t, "qtd_tratativas entre 1 e 3", fato, F.col("qtd_tratativas").between(1, 3))
dq_check(t, "qtd_tratativas = tratativas da silver.tratativas_atendimento", fato_trat,
         F.col("qtd_tratativas") == F.coalesce(F.col("n_trat"), F.lit(0)))

# Pares assunto x subassunto e pico de julho de 2025
dq_check_contagem(t, "pares assunto x subassunto = 17", fato.select("assunto", "subassunto").distinct(), 17)
dq_check_valor(t, "chamados de 2025-07 = 18.061 (pico)", n_fato, fato.filter("mes_referencia = '2025-07'").count(), 18_061)

# Reconciliação com a Silver: a Gold não perdeu nem alterou nada
dq_check_valor(t, "chamados = silver.atendimentos", n_fato, n_fato, n_atend)
dq_check_valor(t, "soma de tempo_resolucao_horas = silver.atendimentos", n_fato,
               fato.agg(F.sum("tempo_resolucao_horas")).first()[0], atend.agg(F.sum("tempo_resolucao_horas")).first()[0])

# Informativo: subassuntos que aparecem em mais de um assunto (esperado: só VISTORIA, em 3 assuntos)
(fato.groupBy("subassunto").agg(F.countDistinct("assunto").alias("assuntos"))
     .filter("assuntos > 1").show(truncate=False))

[PASS] gold.fato_atendimento | total de chamados = 318.871 | 318.871 linhas (esperado 318.871)
[PASS] gold.fato_atendimento | protocolo único | 0 chaves duplicadas de 318871 linhas
[PASS] gold.fato_atendimento | protocolo não nulo | 0/318871 linhas falharam
[PASS] gold.fato_atendimento | id_uc não nulo | 0/318871 linhas falharam
[PASS] gold.fato_atendimento | municipio não nulo | 0/318871 linhas falharam
[PASS] gold.fato_atendimento | bairro não nulo | 0/318871 linhas falharam
[PASS] gold.fato_atendimento | classe não nulo | 0/318871 linhas falharam
[PASS] gold.fato_atendimento | canal não nulo | 0/318871 linhas falharam
[PASS] gold.fato_atendimento | assunto não nulo | 0/318871 linhas falharam
[PASS] gold.fato_atendimento | subassunto não nulo | 0/318871 linhas falharam
[PASS] gold.fato_atendimento | data_hora_abertura não nulo | 0/318871 linhas falharam
[PASS] gold.fato_atendimento | mes_referencia não nulo | 0/318871 linhas falharam
[PASS] gold.fato_atendimento | aberto não nulo | 0

## 6. Registro das checagens em `silver.dq_log`

O resultado é gravado **sempre**, mesmo quando alguma checagem falha, no mesmo log das Silvers e das outras Golds (`table_name = gold.fato_atendimento`). Modo `append`; para ver só a execução atual, filtre pela `checked_at` mais recente.

In [0]:
# Persistência do dq_log (modo append); sempre grava, inclusive quando há falha
df_dq_log = spark.createDataFrame(dq_results)
df_dq_log.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(tabela_dq_log)

display(
    spark.table(tabela_dq_log)
    .filter(F.col("table_name") == "gold.fato_atendimento")
    .orderBy(F.col("checked_at").desc())
    .limit(len(dq_results))
)

table_name,check_name,total_rows,failed_rows,passed,checked_at
gold.fato_atendimento,soma de tempo_resolucao_horas = silver.atendimentos,318871,0,true,2026-10-27T21:50:35.756Z
gold.fato_atendimento,chamados = silver.atendimentos,318871,0,true,2026-10-27T21:50:34.599Z
gold.fato_atendimento,chamados de 2025-07 = 18.061 (pico),318871,0,true,2026-10-27T21:50:34.599Z
gold.fato_atendimento,pares assunto x subassunto = 17,17,0,true,2026-10-27T21:50:34.097Z
gold.fato_atendimento,qtd_tratativas = tratativas da silver.tratativas_atendimento,318871,0,true,2026-10-27T21:50:33.470Z
gold.fato_atendimento,qtd_tratativas entre 1 e 3,318871,0,true,2026-10-27T21:50:31.857Z
gold.fato_atendimento,chamados sem nota = 9.566,318871,0,true,2026-10-27T21:50:30.908Z
gold.fato_atendimento,nota entre 1 e 5 (quando preenchida),318871,0,true,2026-10-27T21:50:30.367Z
gold.fato_atendimento,datas com horário (há aberturas com hora e minuto),318871,0,true,2026-10-27T21:50:29.458Z
gold.fato_atendimento,tempo_resolucao_horas >= 0 e preenchido nos chamados fechados,318871,0,true,2026-10-27T21:50:28.892Z


## 7. Gravação da Gold, só se todas as checagens passarem

Se **qualquer** checagem falhar, a tabela final **não é gravada**: o notebook imprime quais falharam e é interrompido com erro. A `_stg` fica para inspeção, e a `fato_atendimento` anterior, se existir, permanece como estava. Se todas passarem, a final é gravada a partir da `_stg`, e a `_stg` é apagada.

In [0]:
falhas = [r for r in dq_results if not r.passed]

if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_gold} NÃO foi gravada ({tabela_stg} ficou para inspeção):")
    for r in falhas:
        print(f"  - {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"{tabela_gold} não gravada: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

(
    fato.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_gold)
)
spark.sql(f"DROP TABLE IF EXISTS {tabela_stg}")
print(f"Todas as {len(dq_results)} checagens passaram. {tabela_gold} gravada: {n_br(spark.table(tabela_gold).count())} linhas")

Todas as 36 checagens passaram. v_energy.gold.fato_atendimento gravada: 318.871 linhas


## 8. Teste de conexão e consultas de conferência

1. **Conexão com a `dim_uc`:** toda UC com chamado existe na `gold.dim_uc` (e as tabelas concordam em município, bairro e classe).
2. **Visão 360 (Camila):** os chamados de uma UC, do mais recente para o mais antigo, com assunto, canal, tempo e nota.
3. **O pico de julho de 2025 e o diagnóstico:** chamados por mês, comparação dos dois períodos e julho contra os demais meses.
4. **Perguntas do agente (P10):** chamados por assunto e nota média por canal.

In [0]:
f = spark.table(tabela_gold)

# 1. Conexão com a dim_uc (só roda se a gold.dim_uc existir neste workspace)
try:
    dim = spark.table(f"{gold_schema}.dim_uc").select("id_uc", F.col("municipio").alias("m_d"), F.col("bairro").alias("b_d"), F.col("classe").alias("c_d"))
    ucs_f = f.select("id_uc", "municipio", "bairro", "classe").distinct()
    print("UCs com chamado fora da gold.dim_uc:", n_br(ucs_f.join(dim, "id_uc", "left_anti").count()))   # esperado: 0
    dif = ucs_f.join(dim, "id_uc").filter(~(F.col("municipio").eqNullSafe(F.col("m_d")) & F.col("bairro").eqNullSafe(F.col("b_d"))
                                            & F.col("classe").eqNullSafe(F.col("c_d"))))
    print("UCs com município, bairro ou classe diferente da dim_uc:", n_br(dif.count()))                  # esperado: 0
except Exception:
    print("gold.dim_uc não existe neste workspace: a conexão fica para quem tiver as duas tabelas")

# 2. Visão 360: os chamados de uma UC (a que mais abriu)
uc_exemplo = f.groupBy("id_uc").count().orderBy(F.desc("count"), "id_uc").limit(1).first()["id_uc"]
print("UC de exemplo:", uc_exemplo)
display(f.filter(F.col("id_uc") == uc_exemplo)
        .select("protocolo", "data_hora_abertura", "assunto", "subassunto", "canal", "tempo_resolucao_horas", "ultima_acao", "nota_satisfacao")
        .orderBy(F.desc("data_hora_abertura")))

UCs com chamado fora da gold.dim_uc: 0
UCs com município, bairro ou classe diferente da dim_uc: 0
UC de exemplo: UC131266


protocolo,data_hora_abertura,assunto,subassunto,canal,tempo_resolucao_horas,ultima_acao,nota_satisfacao
AT000283299,2026-01-23T13:43:00.000Z,LIGACAO NOVA,SOLICITACAO,PRESENCIAL,178.00,ENCERRAMENTO,4.4
AT000282611,2026-01-11T01:50:00.000Z,RELIGACAO,URGENCIA,WHATSAPP,20.00,ENCERRAMENTO,4.9
AT000264792,2025-10-15T05:20:00.000Z,DANO ELETRICO,RESSARCIMENTO,WHATSAPP,57.00,DESPACHO,4.6
AT000257223,2025-09-03T02:39:00.000Z,GERACAO DISTRIBUIDA,HOMOLOGACAO,TELEFONE,49.00,ABERTURA,4.0
AT000211223,2025-04-21T00:29:00.000Z,LIGACAO NOVA,SOLICITACAO,WHATSAPP,68.00,DESPACHO,3.6
AT000211910,2025-04-05T09:59:00.000Z,GERACAO DISTRIBUIDA,VISTORIA,SITE,160.00,ANALISE,4.1
AT000170046,2024-09-16T14:33:00.000Z,LIGACAO NOVA,PRAZO,APP,116.00,ENCERRAMENTO,4.5
AT000154009,2024-07-13T22:15:00.000Z,DANO ELETRICO,RESSARCIMENTO,SITE,177.00,ABERTURA,4.0
AT000148242,2024-06-26T20:31:00.000Z,RELIGACAO,APOS PAGAMENTO,PRESENCIAL,111.00,DESPACHO,3.9
AT000124247,2024-02-23T00:12:00.000Z,DANO ELETRICO,VISTORIA,WHATSAPP,227.00,ANALISE,4.1


In [0]:
# 3. Chamados por mês e os dois períodos do diagnóstico
# esperado: ~6.400 por mês (de 6.237 a 6.624), exceto 2025-07 = 18.061
f.groupBy("mes_referencia").count().orderBy("mes_referencia").show(60)

p0 = f.filter(F.col("mes_referencia").between("2022-07", "2023-06")).count()
p1 = f.filter(F.col("mes_referencia").between("2025-07", "2026-06")).count()
print(f"primeiro ano: {n_br(p0)} | últimos 12 meses: {n_br(p1)} | variação: {(p1 / p0 - 1) * 100:.1f}%")   # esperado: ~+16%

# julho de 2025 x demais meses (esperado: ~361 h contra ~120 h até fechar; nota ~3,11 contra ~4,10)
(f.withColumn("grupo", F.when(F.col("mes_referencia") == "2025-07", "2025-07").otherwise("demais"))
  .groupBy("grupo")
  .agg(F.count("*").alias("chamados"),
       F.round(F.avg(F.col("tempo_resolucao_horas").cast("double")), 1).alias("horas_ate_fechar"),
       F.round(F.avg(F.col("nota_satisfacao").cast("double")), 2).alias("nota_media"))
  .show())

# 4. Perguntas do agente: chamados por assunto e nota média por canal
# esperado: FALTA DE ENERGIA com 101.306; os outros cinco assuntos entre 43.340 e 43.624
f.groupBy("assunto").count().orderBy(F.desc("count")).show(truncate=False)
f.groupBy("canal").agg(F.count("*").alias("chamados"), F.round(F.avg(F.col("nota_satisfacao").cast("double")), 2).alias("nota_media")).orderBy("canal").show()

+--------------+-----+
|mes_referencia|count|
+--------------+-----+
|       2022-07| 6322|
|       2022-08| 6474|
|       2022-09| 6309|
|       2022-10| 6326|
|       2022-11| 6467|
|       2022-12| 6377|
|       2023-01| 6323|
|       2023-02| 6448|
|       2023-03| 6315|
|       2023-04| 6482|
|       2023-05| 6356|
|       2023-06| 6362|
|       2023-07| 6449|
|       2023-08| 6457|
|       2023-09| 6320|
|       2023-10| 6555|
|       2023-11| 6348|
|       2023-12| 6513|
|       2024-01| 6325|
|       2024-02| 6425|
|       2024-03| 6494|
|       2024-04| 6392|
|       2024-05| 6414|
|       2024-06| 6382|
|       2024-07| 6456|
|       2024-08| 6304|
|       2024-09| 6348|
|       2024-10| 6470|
|       2024-11| 6342|
|       2024-12| 6398|
|       2025-01| 6422|
|       2025-02| 6471|
|       2025-03| 6312|
|       2025-04| 6237|
|       2025-05| 6319|
|       2025-06| 6334|
|       2025-07|18061|
|       2025-08| 6504|
|       2025-09| 6477|
|       2025-10| 6369|
|       202